In [ ]:
import h5py
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
file = r"C:\DATA\StonyBrookCollab\2025_11_05_tpx\prop_oxide_coarse_2025-11-05_16-34.cv4"

data = {}
with h5py.File(file, 'r') as f:
    for key in f.keys():
        data[key] = f[key][:]

df = pd.DataFrame({"x": data['x'], "y": data['y'], "t": data['t'], "pulse": data['cluster_corr']})

In [ ]:
t_threshold = 4000
px.histogram(df[(df['t'] < 10000) & (df['t'] > 0)], x="t").add_vline(
        x=t_threshold, line_dash="dash", line_color="red"
).show()

df['type'] = df['t'].apply(
        lambda t: 'electron' if 3000 < t < t_threshold else 'ion' if t_threshold < t < 8000 else 'background')

In [ ]:
px.density_heatmap(df[df['type'] == "electron"].sample(frac=0.1), x='x', y='y', nbinsx=256, nbinsy=256).show()

In [ ]:
calibration_points = [(4.664, 1), (4.828, 2), (5.77, 18), (5.621, 16), (7.713, 80)]
x_cal, y_cal = zip(*calibration_points)
x_cal = np.array(x_cal) - 4.664 + 4.02

fit = np.polyfit(x_cal, y_cal, 2)
p = np.poly1d(fit)

electrons = df[df['type'] == 'electron'].copy()
ions = df[df['type'] == 'ion'].copy()

ions['mq'] = p(ions['t'] * 1e-3)
px.histogram(ions, x="mq", nbins=1000, title="Ion mass/charge spectrum", log_y=True).show()
spectrum, edges = np.histogram(ions['mq'], bins=1000, range=(0, 100))

smoothed = pd.Series(spectrum).rolling(window=5, center=True).mean()
px.line(x=edges[:-1], y=smoothed, title="Smoothed Ion mass/charge spectrum").show()

In [ ]:
ion_gates = {
    "CHn+": (12, 20),
    "C2H4+": (26, 36),
    "C2H3O+": (38, 50),
    "C3H6O+": (56, 70),
}
for species, (low, high) in ion_gates.items():
    gate_ions = ions[(ions['mq'] > low) & (ions['mq'] < high)]
    px.density_heatmap(
            gate_ions, x="x", y="y", nbinsx=256, nbinsy=256,
            title=f"{species} spatial distribution"
    ).show()
    ions['species'] = np.where(
            (ions['mq'] > low) & (ions['mq'] < high),
            species,
            ions.get('species', 'unknown')
    )

ion_center = 145, 133
ions['high_KER'] = np.sqrt((ions['x'] - ion_center[0]) ** 2 + (ions['y'] - ion_center[1]) ** 2) > 20



for species in ions['species'].unique():
    for ker in [True, False]:
        subset= ions[(ions['species']==species)&(ions['high_KER']==ker)]
        px.density_heatmap(
            subset, x="x", y="y", nbinsx=256, nbinsy=256,
            title=f"{species} spatial distribution, high KER={ker}"
        ).show()

shot_df=df.groupby('pulses').agg(
    total_counts=('t','count'),
    electron_counts=('type', lambda x: (x=='electron').sum()),
    ion_counts=('type', lambda x: (x=='ion').sum()),
)
px.histogram(shot_df, x="ion_counts", nbins=100, title="Ion counts per shot").show()
px.histogram(shot_df, x="electron_counts", nbins=100, title="Electron counts per shot").show()


In [ ]:
electron_center = 138, 133

for species, (low, high) in ion_gates.items():
    for high_ker in [True, False]:
        e_filt = electrons.merge(
                ions.loc[(ions['species'] == species) & (ions['high_KER'] == high_ker), ["pulse"]],
                on='pulse',
                how='inner'
        )

        e_filt['r'] = np.sqrt((e_filt['x'] - electron_center[0]) ** 2 + (e_filt['y'] - electron_center[1]) ** 2)

        px.histogram(
                e_filt,
                x="r",
                title=f"{species} spatial distribution, {high_ker} high KER"
        ).show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

electrons['r'] = np.sqrt((electrons['x'] - electron_center[0]) ** 2 + (electrons['y'] - electron_center[1]) ** 2)
bins = np.linspace(0, 100, 50)
n_shots = df['pulse'].max() + 1

for species, (low, high) in ion_gates.items():
    for high_ker in [True, False]:
        # 1) electron data (p, trigindex1)
        e = electrons.copy()
        p = e['r'].to_numpy()
        trigindex1 = e['pulse'].to_numpy()

        # 2) ion weights per shot (weight[shot] = # ions in that shot after gating)
        pulse_counts = (
            ions.loc[(ions['species'] == species) & (ions['high_KER'] == high_ker), 'pulse']
            .value_counts()
        )

        # IMPORTANT: match cov1D's n_shots definition
        n_shots = int(trigindex1.max()) + 1

        # reindex to include zero-ion shots
        weight = pulse_counts.reindex(np.arange(n_shots), fill_value=0).to_numpy()

        # 3) bin indices (match digitize(..., right=True)-1)
        bin_idx = np.digitize(p, bins, right=True) - 1
        in_range = (bin_idx >= 0) & (bin_idx < len(bins) - 1)

        # 4) EXY accumulation: each electron contributes weight[its_shot]
        EXY = np.bincount(
                bin_idx[in_range],
                weights=weight[trigindex1[in_range]],
                minlength=len(bins) - 1
        ).astype(float)

        # 5) EXEY and normalization (same as cov1D)
        EXEY = np.histogram(p, bins)[0].astype(float)

        EXY /= n_shots
        EXEY *= (weight.sum() / (n_shots ** 2.0))

        fig = make_subplots(rows=1, cols=2, subplot_titles=("EXY, EXEY", "Covariance"))
        fig.add_trace(
                go.Scatter(x=bins[:-1], y=EXY, mode='lines', name='EXY'),
                row=1, col=1
        )
        fig.add_trace(
                go.Scatter(x=bins[:-1], y=EXEY, mode='lines', name='EXEY'),
                row=1, col=1
        )
        fig.add_trace(
                go.Scatter(x=bins[:-1], y=EXY - EXEY, mode='lines', name='Covariance'),
                row=1, col=2
        )

        fig.show()

        break
    break